# Swarm Clock, Part 3a: sampling-matched power (upper bound), NCD ablation, configuration sensitivity
Same runtime, after 2d (uses REAL, WIN, window_test, rho_window, rho_window_xlabel, fam_pairs, mutate, vocab, reps_with_strata, D_primary, elig).

**Why:** a negative result is only as strong as the test's power under the *real* sampling design (Murray et al., 2016). We plant accumulating lineages that reuse each real family's exact timestamps, labels and size, then ask how often the frozen long-lag test detects them. This yields the smallest per-copy edit rate we can exclude.

In [ ]:
import zlib
X_primary = reps_with_strata(D_primary, elig.index, "length")
REAL_FAMS = {f: X_primary[X_primary.family == f].sort_values("t").reset_index(drop=True) for f in X_primary.family.unique()}
REAL_FAMS = {f: sub for f, sub in REAL_FAMS.items() if sub.t.dt.date.nunique() >= 2}   # families with any long-lag pairs
print({f: (len(s), s.t.dt.date.nunique()) for f, s in REAL_FAMS.items()})

def synth_like(sub, p, window=5):
    toks0 = sub.m.iloc[0].split()          # root = the family's earliest real introduction
    members = []
    for i in range(len(sub)):
        parent = toks0 if i == 0 else members[rng.integers(max(0, i - window), i)]
        members.append(mutate(parent, p))
    return sub.assign(m=[" ".join(x) for x in members])

def pooled_power(p, reps=20, B=99, lo=6, hi=1e9):
    hits_all, hits_x = [], []
    for _ in range(reps):
        Ps = [fam_pairs(synth_like(sub, p)) for sub in REAL_FAMS.values()]
        hits_all.append(window_test(Ps, lo, hi, B=B)[1] < .05)
        globals()["rho_window"], keep = rho_window_xlabel, rho_window
        hits_x.append(window_test(Ps, lo, hi, B=B)[1] < .05)
        globals()["rho_window"] = keep
    return np.mean(hits_all), np.mean(hits_x)

def drift_per_day(p, reps=5):
    v = []
    for _ in range(reps):
        for sub in REAL_FAMS.values():
            P = fam_pairs(synth_like(sub, p)); lag = np.abs(P["t"][P["i"]] - P["t"][P["j"]]) / 24
            m = lag >= 0.25
            if m.sum() > 10: v.append(np.polyfit(lag[m], P["d"][m], 1)[0])
    return np.median(v)
POWER = []
for p in [0.0025, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20]:
    pa, px = pooled_power(p)
    POWER.append({"edit_rate_per_copy": p, "distance_slope_per_day": round(drift_per_day(p), 4),
                  "power_long_lag_all_pairs": pa, "power_long_lag_cross_label": px})
POWER = pd.DataFrame(POWER); print(POWER.to_string(index=False)); POWER.to_csv("data/power_sampling_matched.csv", index=False)
# Observed long-lag distance slope in real families, for comparison
obs_slopes = []
for f, sub in REAL_FAMS.items():
    P = fam_pairs(sub); lag = np.abs(P["t"][P["i"]] - P["t"][P["j"]]) / 24; m = lag >= 0.25
    if m.sum() > 10: obs_slopes.append((f, round(np.polyfit(lag[m], P["d"][m], 1)[0], 4)))
print("observed long-lag distance slopes per day:", obs_slopes)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5,3.6))
ax.plot(POWER.edit_rate_per_copy, POWER.power_long_lag_all_pairs, "-o", label="all pairs")
ax.plot(POWER.edit_rate_per_copy, POWER.power_long_lag_cross_label, "-s", label="cross-label pairs")
ax.axhline(.8, ls=":", c="gray"); ax.set_xscale("log"); ax.set_xlabel("per-copy edit rate of a planted clock (log)")
ax.set_ylabel("power of long-lag test (>= 6 h)"); ax.set_title("Sampling-matched power", fontsize=10); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("fig_power.png", dpi=200); plt.show()

## 2. Distance ablation: normalized compression distance (zlib) instead of shingle Jaccard

In [ ]:
def ncd(a, b):
    ca, cb, cab = len(zlib.compress(a.encode(), 9)), len(zlib.compress(b.encode(), 9)), len(zlib.compress((a + " " + b).encode(), 9))
    return (cab - min(ca, cb)) / max(ca, cb)
def fam_pairs_ncd(sub):
    P = fam_pairs(sub); txt = sub.m.values
    P["d"] = np.array([ncd(txt[a], txt[b]) for a, b in zip(P["i"], P["j"])]); return P
ABL = []
Ps_ncd = [fam_pairs_ncd(X_primary[X_primary.family == f]) for f in X_primary.family.unique()]
for cross in [False, True]:
    keep = rho_window
    if cross: globals()["rho_window"] = rho_window_xlabel
    for wname, lo, hi in WIN:
        T, p, v = window_test(Ps_ncd, lo, hi)
        ABL.append({"distance": "NCD", "pairs": "cross-label" if cross else "all", "window": wname, "T": round(T,3) if T==T else T, "p": p})
    globals()["rho_window"] = keep
print(pd.DataFrame(ABL).to_string(index=False))

## 3. Configuration sensitivity: the same window tests with families from (k=5, threshold 0.3)

In [ ]:
fam_k5 = assign_families(real_minhashes(5), 0.3)
D_k5, G_k5 = family_table(R, fam_k5)
elig_k5 = G_k5[G_k5.blocks >= 15].index
X_k5 = reps_with_strata(D_k5, elig_k5, "length")
Ps_k5 = [fam_pairs(X_k5[X_k5.family == f]) for f in X_k5.family.unique()]
for cross in [False, True]:
    keep = rho_window
    if cross: globals()["rho_window"] = rho_window_xlabel
    for wname, lo, hi in WIN:
        T, p, v = window_test(Ps_k5, lo, hi)
        ABL.append({"distance": "Jaccard k5 t0.3", "pairs": "cross-label" if cross else "all", "window": wname, "T": round(T,3) if T==T else T, "p": p})
    globals()["rho_window"] = keep
ABL = pd.DataFrame(ABL); print(ABL.to_string(index=False)); ABL.to_csv("data/ablations.csv", index=False)